<a href="https://colab.research.google.com/github/AlbertPuentes/Machine-Learning_Semana_12_Actvidad_RRHH./blob/main/CasoRRHH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CASO DE ESTUDIO: Predicción de defectos en Producción

La empresa manufacturera XYZ Inc desea predecir si un lote de producción saldrá  efectuoso o no defectuoso, usando variables del proceso productivo. La variable objetivo será cero (0.0) si el lote sale no defectuoso y uno “1” para un lote defectuoso.

Es importante además tener en cuenta variables como: temperatura
(temperatura de la máquina), presión (la presión del sistema), velocidad
(velocidad de producción), hora de operación (horas continuas del uso de
la máquina), vibración (nivel de vibración de la máquina).

Para una empresa del sector de producción, la generación de un lote
defectuoso puede traer como consecuencias: pérdidas económicas,
reprocesos, devoluciones, desperdicio de materia prima, afectación de la
calidad, entre otras.

En este caso, la métrica recall podría resultar más interesante de
estudiar porque queremos detectar la mayor cantidad posible de lotes
defectuosos. Sin embargo, es importante estudiar todas las métricas.


In [1]:
# Predicción de rotación de empleados
# ============================================================

import pandas as pd

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Evaluación
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. DATASET
# ============================================================

# renuncia:
# 0 = permanece
# 1 = renuncia

data = {
    "edad": [
        25, 28, 35, 40, 30, 45, 32, 29, 50, 38,
        27, 33, 41, 36, 31, 48, 26, 39, 42, 34
    ],
    "salario": [
        1800, 2000, 3500, 4500, 2200, 5000, 2800, 2100, 6000, 4000,
        1900, 3000, 4700, 3600, 2500, 5500, 1700, 4200, 4800, 3200
    ],
    "años_empresa": [
        1, 2, 5, 8, 2, 10, 4, 2, 15, 7,
        1, 4, 9, 6, 3, 12, 1, 8, 10, 5
    ],
    "satisfaccion": [
        2, 3, 7, 8, 3, 9, 5, 2, 9, 7,
        2, 6, 8, 6, 4, 9, 1, 7, 8, 5
    ],
    "horas_extra": [
        15, 12, 5, 4, 14, 3, 8, 13, 2, 6,
        16, 7, 4, 6, 10, 3, 18, 5, 4, 8
    ],
    "renuncia": [
        1, 1, 0, 0, 1, 0, 0, 1, 0, 0,
        1, 0, 0, 0, 1, 0, 1, 0, 0, 0
    ]
}

df = pd.DataFrame(data)

print("Dataset inicial:")
print(df.head())

# ============================================================
# 2. VARIABLES X e y
# ============================================================

X = df[
    [
        "edad",
        "salario",
        "años_empresa",
        "satisfaccion",
        "horas_extra"
    ]
]

y = df["renuncia"]

# ============================================================
# 3. DIVISIÓN DE DATOS
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# ============================================================
# 4. MODELOS
# ============================================================

modelos = {
    "Regresión Logística": LogisticRegression(max_iter=1000),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "SVM": SVC(kernel="linear"),
    "Naive Bayes": GaussianNB()
}

# ============================================================
# 5. EVALUACIÓN DE MODELOS
# ============================================================

resultados = []

for nombre, modelo in modelos.items():

    print("\n" + "="*70)
    print("MODELO:", nombre)
    print("="*70)

    # Entrenamiento
    modelo.fit(X_train, y_train)

    # Predicción
    y_pred = modelo.predict(X_test)

    # Métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Validación cruzada
    cv_scores = cross_val_score(
        modelo,
        X,
        y,
        cv=5,
        scoring="accuracy"
    )

    cv_promedio = cv_scores.mean()

    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Validación cruzada": cv_promedio
    })

    print("\nMatriz de confusión:")
    print(confusion_matrix(y_test, y_pred))

    print("\nReporte de clasificación:")
    print(classification_report(y_test, y_pred, zero_division=0))

    print("Validación cruzada:", cv_scores)
    print("Promedio:", cv_promedio)

# ============================================================
# 6. TABLA FINAL
# ============================================================

df_resultados = pd.DataFrame(resultados)

print("\n" + "="*70)
print("TABLA COMPARATIVA")
print("="*70)

print(df_resultados.sort_values(by="F1-score", ascending=False))

# ============================================================
# 7. MEJOR MODELO
# ============================================================

mejor_modelo = df_resultados.sort_values(
    by="F1-score",
    ascending=False
).iloc[0]

print("\nMejor modelo según F1-score:")
print(mejor_modelo)

Dataset inicial:
   edad  salario  años_empresa  satisfaccion  horas_extra  renuncia
0    25     1800             1             2           15         1
1    28     2000             2             3           12         1
2    35     3500             5             7            5         0
3    40     4500             8             8            4         0
4    30     2200             2             3           14         1

MODELO: Regresión Logística

Matriz de confusión:
[[4 0]
 [0 2]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00         2

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

Validación cruzada: [1. 1. 1. 1. 1.]
Promedio: 1.0

MODELO: Árbol de Decisión

Matriz de confusión:
[[4 0]
 [0 2]]

Reporte de clasificación:
             

# Análisis de los Resultados caso de estudio de RRHH

## 1. Interpretar: FN, FP, TP, TN
En la matriz de confusión se evalúo los aciertos y errores del modelo comparando las predicciones con la realidad: 1 = Renuncia, 0 = Permanece

* **TP (Verdadero Positivo):** El modelo predijo que el empleado renunciaría 1, y el empleado renunció.
* **TN (Verdadero Negativo):** El modelo predijo que el empleado se quedaría 0, y el empleado permaneció en la empresa.
* **FP (Falso Positivo):** El modelo predijo que el empleado renunciaría 1, pero en realidad decidió quedarse 0. Es una falsa alarma.
* **FN (Falso Negativo):** El modelo predijo que el empleado se quedaría 0, pero sorpresivamente renunció 1. Es el error más crítico en este escenario lo que genera costos imprevistos.

## 2. ¿Por qué es importante predecir la rotación de empleados?
Predecir la rotación permite a la gerencia pasar de ser reactiva a proactiva. Facilita la identificación de talento en riesgo antes de que se presente una carta de renuncia, brindando a Recursos Humanos una ventana de tiempo para intervenir con mejores condiciones, planes de carrera o ajustes, logrando así retener al personal clave.
Reducción de costos: Evita gastos de reclutamiento y capacitación.
Preservación del conocimiento: Se Minimiza la pérdida de experiencia.
Productividad: Mantiene la continuidad operativa y evita curvas de aprendizaje.
Clima laboral: Reduce el impacto negativo en la moral del equipo.

## 3. ¿Qué impacto económico puede generar la renuncia de personal?
La fuga de talento tiene un costo elevado para la organización, dividido en dos categorías:
* **Costos Directos:**
• Reclutamiento: anuncios, headhunters, tiempo de selección
• Onboarding: capacitación, mentoría, materiales
• Productividad perdida: tiempo de capacitacion para alcanzar pleno rendimiento

* **Costos Indirectos:**
• Sobrecarga al equipo restante, más rotación
• Pérdida de relaciones con clientes, proveedores
• Impacto en reputación marca empleadora

## 4. ¿Por qué este problema corresponde a un problema de clasificación supervisada?
Este caso corresponde a un problema supervisado porque alimentamos al algoritmo con datos históricos que ya poseen una etiqueta de respuesta correcta la columna, renuncia.
Es de clasificación porque la variable objetivo es categórica y discreta las opciones son excluyentes: 0 = permanece, 1 = renuncia, a diferencia de la regresión que estima valores numéricos continuos.

## 5. Entre las variables (edad, salario, años de permanencia en la empresa, nivel de satisfacción, horas extras) ¿cuáles de ellas tendrán mayor influencia sobre la renuncia de un empleado y por qué?

### Variables con mayor influencia
De acuerdo al dataset, las variables más determinantes son:
1. **Nivel de satisfacción:** Refleja el clima y estado emocional. Valores bajos suelen correlacionarse con renuncias.
2. **Horas extras:** Indican sobrecarga y riesgo de *burnout*.
3. **salario:** Salarios por debajo del mercado incentivan la búsqueda externa
4. **edad** perfiles jóvenes con salarios bajos tienden a rotar con mayor frecuencia en busca de nuevas oportunidades.

## 6. Agregue al modelo dos variables (entre: clima laboral, liderazgo, modalidad de trabajo, evaluación de desempeño). ¿Considera ud que el modelo predictivo podría mejorar? ¿Por qué si y por qué no?

Sí, el modelo podría mejorar sustancialmente. Variables como el liderazgo son razones primarias de renuncia a nivel global, y la modalidad de trabajo aporta contexto actual de gran valor.

**Posibles limitaciones**:
Dificultad para medir objetivamente.
Riesgo de multicolinealidad con satisfacción.
Requieren recolección de datos adicional.

## 7. Al ejecutar el algoritmo, ¿cuál sería la interpretación de cada una de las métricas?
* **Accuracy (Exactitud):** Total de predicciones correctas. Engañosa si hay desbalance de clases.
* **Precision (Precisión):** La calidad de los positivos. De todos los que predijo que renunciarían, ¿cuántos realmente lo hicieron?. Penaliza los Falsos Positivos.
* **Recall (Sensibilidad):** La capacidad de hallazgo. De todos los que realmente renunciaron, ¿cuántos logró detectar el modelo?. Penaliza los Falsos Negativos.
* **F1-Score:** La media armónica entre Precisión y Recall. Ideal para evaluar modelos con datos desbalanceados.

## 8. Suponga que uno de los modelos obtuvo: Accuracy = 0.90, precisión = 0.70, recall = 0.95, F1-Score = 0.80. ¿Es un buen modelo?
Es un excelente modelo para Recursos Humanos. Aunque la Precisión del 0.70 indica que genera algunas falsas alarmaspredice que alguien se va cuando en realidad se queda, el Recall de 0.95 indica que es capaz de detectar al 95% de las personas que realmente se van a ir, el F1-Score 0.80 indica que es equilibrado lo que da un balance entre no alarmar innecesariamente y no perder casos reales.

## 9. ¿Cuál de las métricas considera más importante y cuál sería el motivo?
La métrica más importante para este caso es Recall.
Motivo: es preferible lidiar con un Falso Positivo que sufrir un Falso Negativo.
En RRHH, el costo de un Falso Negativoes mucho mayor que el de un Falso Positivo.

## 10. Si el modelo presenta muchos “Falsos Negativos”, ¿qué consecuencias podría tener para la empresa?

###Consecuencias de los Falsos Negativos
Si existen muchos Falsos Negativos, la gerencia tendría una falsa sensación de seguridad. El modelo predice que el personal es estable, pero habra renuncias que seguirían ocurriendo sorpresivamente. De esta manera el algoritmo perdería credibilidad y la empresa seguiría absorbiendo los altos costos de rotación.

## 11. Después de ejecutar el algoritmo con los diferentes modelos,¿cuál fue el mejor modelo según la métrica F1-Score? ¿Por qué cree que este modelo tuvo el mejor desempeño?

El mejor modelo según F1-Score.
Al ejecutar los algoritmos con el dataset educativo adjunto, modelos como Regresión Logística, Árboles de Decisión y Random Forest obtienen un F1-Score (1.0). Se debe a que el dataset de prueba es extremadamente pequeño apenas 20 registros y las variables clave como el nivel de satisfacción y las horas extras separan a los empleados que renuncian de los que se quedan de una manera muy clara y evidente. En un entorno real con miles de registros y datos se genera mas ruido, lo cual ocasiona que los resultados varien y modelos como Random Forest o SVM destacarían por encima de los más simples.

¿Por qué tuvo el mejor desempeño?
No fue por superioridad algorítmica, sino por un dataset demasiado pequeño, patrones linealmente separables en la muestra, y azar en la partición de datos. En un entorno real con más datos, Random Forest o Gradient Boosting suelen superar a LR en problemas no lineales, mientras que LR sigue siendo preferible por interpretabilidad.

## 12. Aunque Random Forest tenga mejor Accuracy, ¿en qué escenario sería más conveniente utilizar un Árbol de Decisión?
Aunque un Random Forest suele ser más exacto, se elegiría un Árbol de Decisión cuando la interpretabilidad sea prioritaria. Random Forest funciona como una caja negra en contraste, un Árbol de Decisión simple permite extraer reglas de negocio claras para presentar.
En datasets pequeños, un árbol de Decisión puede generalizar mejor que un ensemble que busca patrones complejos.

## 13. Desde el punto de vista de la ética, ¿Considera ético utilizar Machine Learning para predecir la renuncia de empleados?

Depende del uso que se le da al ML,  es ético si los datos utilizados se manejan en totales con el fin de mantener en minimos la tasa de renincias, generando estrategias que aumenten lamotivacion de los empleados de forma general, y no es ético si las predicciones son utilizadas de forma individual con cada empleado por ejemplo negar ascensos o despedir anticipadamente a un empelado al que el ML predijo tendria posibilidad de una pronta renuncia.

## 14. Si estuviéramos analizando 10.000 empleados y la mayoría NO renuncia, ¿Qué problema podría presentarse?

Si el 95% del personal nunca renuncia, nos enfrentamos a un problema de Desbalanceo de Clases. El modelo podría predecir que nadie renuncia y aún así obtener un Accuracy del 95%, siendo completamente inútil al fallar en detectar al 5% crítico. Requiere el uso de métricas como el F1-Score y técnicas de remuestreo como SMOTE.

## 15. Explique el concepto de “Overfitting”. ¿Esto podría ocurrir en este caso de estudio?

El Overfitting sucede cuando el algoritmo memoriza los datos de entrenamiento incluyendo ruido y patrones aleatorios, perdiendo capacidad de generalizar a nuevos datos. En este caso de estudio es muy seguro que ocurra, ya que entrenar modelos con apenas 20 registros resultaría en la memorización de empleados específicos en lugar de patrones reales.

## 16. Si ud fuera el (la) gerente de RRHH, ¿prefería un modelo más preciso pero difícil de explicar? o ¿preferiría un modelo menos preciso pero fácil de interpretar?
Si fuese Gerente de RRHH, Preferiria un modelo que sea fácil de interpretar aunque sea menos preciso, ya que para justificar cambios en la empresa se hace  necesario explicar el por qué y si el modelo es fácil de interpretar seria mucho mas sencillo realizar esta actividad de lo predicho por el modelo, lo cual no seria sencillo con un modelo mas complejo aunque sea mas preciso.

## 17. ¿Cuál considera es el mejor algoritmo para este caso de estudio? ¿Cuál es la justificación?
Para este dataset educativo en particular, la Regresión Logística asi como un Árbol de Decisión de poca profundidad son las mejores opciones.
Son menos propensos al sobreajuste en muestras pequeñas, altamente interpretables y, dado que la relación de variables es clara, ofrecen resultados perfectos o cuasi-perfectos sin la sobrecarga de computación de modelos más densos.
El Árbol de Decisión es elseria la mejor opción porque logra el equilibrio entre aciertos de las predicciones en este escenario, no requiere un poder computacional alto y traduce los datos en información que cualquier gerente sin conocimientos especiales como en  Machine Learning lo que le permite entender de forma mas sencilla.